# 🕳️ Pothole Detection System — Multi-Stage YOLOv8 Training (Colab)

## 🚀 Fully Automated Sequential Training Pipeline

This notebook is **optimized for Google Colab** with GDrive persistence.

---

| Feature | Detail |
|----------|----------|
| **Environment** | Google Colab (12h session limit) |
| **Stage 1** | YOLOv8m · 768px · 400 epochs |
| **Stage 2** | YOLOv8l · 832px · 400 epochs |
| **Time Guard** | Auto-save before 11.5h limit |
| **Resume** | `last.pt` → `best.pt` → fresh (cascade) |
| **GPU Safety** | Dynamic batch · OOM recovery · imgsz fallback |

## 1 · Environment Setup

In [1]:
"""
Section 1 · Environment Setup — Google Colab
Mounts GDrive, installs deps, detects GPU.
"""

from __future__ import annotations

# ── Environment Detection ──────────────────────────────────────
ENV = "COLAB"
SESSION_TIME_LIMIT_H = 11.5   # Safe margin under Colab's 12h limit
print("=" * 60)
print(f"  [ENV DETECTED] {ENV}")
print(f"  [SESSION LIMIT] {SESSION_TIME_LIMIT_H} hours")
print("=" * 60)

# Install dependencies
!pip install -q ultralytics

import gc, json, os, shutil, time, warnings
from datetime import datetime, timezone
from pathlib import Path

# ── Fix existing drive mount issues ─────────────────────────────
if os.path.exists("/content/drive"):
    try:
        shutil.rmtree("/content/drive")
    except:
        pass

# ── Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted.")

# ── Standard imports ───────────────────────────────────────────
import cv2, matplotlib.pyplot as plt, numpy as np, pandas as pd
import seaborn as sns, torch, yaml
from IPython.display import Image as IPImage, display
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {total_vram:.1f} GB")

  [ENV DETECTED] COLAB
  [SESSION LIMIT] 11.5 hours
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.1 MB/s eta 0:00:00
Mounted at /content/drive
✅ Google Drive mounted.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


## 2 · Project Paths & Dataset

In [2]:
"""
Section 2a · Project Paths (Colab + GDrive)

BASE_DIR = /content (Colab VM local disk)
All persistent storage on GDrive.
"""
from pathlib import Path

# ── Dynamic base ─────────────────────────────────────────────────
BASE_DIR = Path("/content")

# ── GDrive project root ──────────────────────────────────────────
GDRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/pothole_detection_system")
BACKEND_DIR = GDRIVE_PROJECT_ROOT / "backend"
PROJECT_ROOT = GDRIVE_PROJECT_ROOT

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BACKEND_DIR :", BACKEND_DIR)

# ── Dataset source (on GDrive) ──────────────────────────────────
DATASET_DIR_GDRIVE = BACKEND_DIR / "dataset_768"

# ── Local cache for dataset (fast I/O on Colab VM) ──────────────
LOCAL_CACHE_DIR = BASE_DIR / "dataset_768_cache"
DATASET_DIR = LOCAL_CACHE_DIR
DATA_YAML   = DATASET_DIR / "data.yaml"

# ── Model outputs (on GDrive for persistence) ────────────────────
MODELS_DIR      = BACKEND_DIR / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"
FINAL_DIR       = MODELS_DIR / "final"
LOGS_DIR        = MODELS_DIR / "logs"
V8M_DIR         = MODELS_DIR / "v8m"
V8L_DIR         = MODELS_DIR / "v8l"

INFERENCE_DIR = BACKEND_DIR / "inference_outputs"
RUNS_DIR      = BACKEND_DIR / "runs"
UTILS_DIR     = BACKEND_DIR / "utils"

for d in [MODELS_DIR, CHECKPOINTS_DIR, FINAL_DIR, LOGS_DIR,
          V8M_DIR, V8L_DIR, INFERENCE_DIR, RUNS_DIR, UTILS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\n📁 Directory structure:")
for d in [MODELS_DIR, CHECKPOINTS_DIR, FINAL_DIR, LOGS_DIR,
          V8M_DIR, V8L_DIR, INFERENCE_DIR, RUNS_DIR, UTILS_DIR]:
    print("  ✅", d)

print(f"\n📦 Dataset source (GDrive): {DATASET_DIR_GDRIVE}")
print(f"📦 Dataset cache  (local) : {LOCAL_CACHE_DIR}")
assert DATASET_DIR_GDRIVE.exists(), f"❌ Dataset not found on GDrive at {DATASET_DIR_GDRIVE}"

PROJECT_ROOT: /content/drive/MyDrive/pothole_detection_system
BACKEND_DIR : /content/drive/MyDrive/pothole_detection_system/backend

📁 Directory structure:
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/checkpoints
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/final
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/logs
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/v8m
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/models/v8l
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/inference_outputs
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/runs
  ✅ /content/drive/MyDrive/pothole_detection_system/backend/utils

📦 Dataset source (GDrive): /content/drive/MyDrive/pothole_detection_system/backend/dataset_768
📦 Dataset cache  (local) : /content/dataset_768_cache


In [3]:
"""
Section 2b · Smart Dataset Caching (Colab)

1. Use dataset_768.tar if available (fastest)
2. Otherwise copy dataset folder from GDrive
3. Skip if cache already exists
"""
import time as _time

TAR_FILE_GDRIVE = DATASET_DIR_GDRIVE.parent / "dataset_768.tar"
LOCAL_TAR_FILE = BASE_DIR / "dataset_768.tar"

if LOCAL_CACHE_DIR.exists() and (LOCAL_CACHE_DIR / "images").exists():
    print(f"✅ Dataset cache already exists at {LOCAL_CACHE_DIR}")
    print("   Skipping caching step.")
else:
    if LOCAL_CACHE_DIR.exists():
        shutil.rmtree(LOCAL_CACHE_DIR)

    if TAR_FILE_GDRIVE.exists():
        print("📦 TAR dataset detected → using fast extraction method")
        print(f"Source: {TAR_FILE_GDRIVE}")
        t0 = _time.time()
        !cp "{TAR_FILE_GDRIVE}" "{LOCAL_TAR_FILE}"
        print(f"✅ TAR copied in {_time.time()-t0:.1f}s")
        print("\n📦 Extracting dataset...")
        t1 = _time.time()
        !tar -xf "{LOCAL_TAR_FILE}" -C {BASE_DIR}/
        if (BASE_DIR / "dataset_768").exists():
            shutil.move(str(BASE_DIR / "dataset_768"), str(LOCAL_CACHE_DIR))
        print(f"✅ Dataset extracted in {_time.time()-t1:.1f}s")
    else:
        print("⚠️ TAR file not found. Falling back to folder copy.")
        print(f"Source: {DATASET_DIR_GDRIVE}")
        t0 = _time.time()
        !cp -r "{DATASET_DIR_GDRIVE}" "{LOCAL_CACHE_DIR}"
        print(f"✅ Dataset copied in {_time.time()-t0:.1f}s")

# ── Update data.yaml ─────────────────────────────────────────────
DATA_YAML = LOCAL_CACHE_DIR / "data.yaml"
if not DATA_YAML.exists():
    raise FileNotFoundError(f"❌ data.yaml not found in {LOCAL_CACHE_DIR}")

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["path"] = str(LOCAL_CACHE_DIR.resolve())
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print("\n📄 data.yaml updated:")
for k, v in data_cfg.items():
    print(f"  {k}: {v}")

_n_train = len(list((LOCAL_CACHE_DIR / "images" / "train").glob("*.*")))
_n_val = len(list((LOCAL_CACHE_DIR / "images" / "val").glob("*.*")))
_n_test = len(list((LOCAL_CACHE_DIR / "images" / "test").glob("*.*")))

print(f"\n📊 Cached images:")
print(f"   train: {_n_train}")
print(f"   val  : {_n_val}")
print(f"   test : {_n_test}")

print("\n" + "=" * 60)
print("  [DATASET READY] ✅ Dataset cached and verified")
print("=" * 60)

📦 TAR dataset detected → using fast extraction method
Source: /content/drive/MyDrive/pothole_detection_system/backend/dataset_768.tar
✅ TAR copied in 68.8s

📦 Extracting dataset...
✅ Dataset extracted in 21.3s

📄 data.yaml updated:
  path: /content/dataset_768_cache
  train: images/train
  val: images/val
  test: images/test
  nc: 1
  names: ['pothole']

📊 Cached images:
   train: 13284
   val  : 3796
   test : 1896

  [DATASET READY] ✅ Dataset cached and verified


## 3 · GPU Detection & Dynamic Batch Sizing

In [4]:
"""
Dynamic GPU batch sizing via real memory profiling.

Instead of hardcoded heuristics, this cell:
  1. Measures actual per-image GPU memory by running a forward pass with batch=1 & batch=2
  2. Calculates the optimal batch size to use 85-90% of GPU VRAM
  3. Falls back to conservative estimates if profiling fails
"""

# ── Configuration ────────────────────────────────────────────────
GPU_TARGET_UTILIZATION = 0.875   # Target 87.5% of total VRAM (sweet spot of 85-90%)
MIN_BATCH   = 2
MAX_BATCH   = 64
IMGSZ_FALLBACKS = [832, 768, 640, 512]   # Try these in order if needed


def _profile_model_memory(model_name: str, imgsz: int) -> tuple:
    """
    Profile GPU memory usage by running real forward passes.

    Returns:
        (base_memory_gb, per_image_memory_gb)

    base_memory   = GPU memory for model weights + optimizer buffers
    per_image_mem = additional memory per image in a batch (activations + grads)
    """
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats(0)

    mem_before = torch.cuda.memory_allocated(0) / 1e9

    # Load model and move to GPU
    model = YOLO(model_name)
    model.model.to("cuda").train()

    # ── Forward pass with batch=1 ─────────────────────────────
    torch.cuda.reset_peak_memory_stats(0)
    dummy1 = torch.randn(1, 3, imgsz, imgsz, device="cuda", dtype=torch.float32)
    with torch.no_grad():
        _ = model.model(dummy1)
    mem_peak_b1 = torch.cuda.max_memory_allocated(0) / 1e9

    del dummy1, _
    torch.cuda.empty_cache()

    # ── Forward pass with batch=2 ─────────────────────────────
    torch.cuda.reset_peak_memory_stats(0)
    dummy2 = torch.randn(2, 3, imgsz, imgsz, device="cuda", dtype=torch.float32)
    with torch.no_grad():
        _ = model.model(dummy2)
    mem_peak_b2 = torch.cuda.max_memory_allocated(0) / 1e9

    del dummy2, _

    # ── Calculate per-image and base memory ───────────────────
    per_image_mem = mem_peak_b2 - mem_peak_b1  # Delta between batch=2 and batch=1
    base_mem = mem_peak_b1 - per_image_mem      # Base = total_b1 - 1*per_image

    # During actual training with AMP + optimizer + gradients,
    # memory usage is roughly 2-2.5x the forward-only pass.
    # Apply a training overhead multiplier to account for this.
    TRAINING_OVERHEAD = 2.2
    per_image_mem *= TRAINING_OVERHEAD
    base_mem *= TRAINING_OVERHEAD

    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()

    # Sanity: per_image must be positive
    if per_image_mem <= 0:
        per_image_mem = mem_peak_b1 * 0.4
        base_mem = mem_peak_b1 * 0.6

    return max(base_mem, 0.5), max(per_image_mem, 0.05)


def estimate_batch_size(model_name: str, imgsz: int) -> tuple:
    """
    Dynamically estimate the optimal batch size for the given model and image size.

    Strategy:
      1. Profile actual GPU memory using forward passes
      2. Calculate max batch to use GPU_TARGET_UTILIZATION of VRAM
      3. If batch < MIN_BATCH, try smaller image sizes

    Returns:
        (batch_size, actual_imgsz) — imgsz may be reduced if GPU is too small.
    """
    if not torch.cuda.is_available():
        print("⚠️  No GPU — defaulting to batch=2, CPU mode.")
        return MIN_BATCH, imgsz

    props = torch.cuda.get_device_properties(0)
    total_vram = props.total_memory / 1e9
    target_vram = total_vram * GPU_TARGET_UTILIZATION

    print(f"🖥️  GPU: {props.name}")
    print(f"   Total VRAM : {total_vram:.2f} GB")
    print(f"   Target usage: {target_vram:.2f} GB ({GPU_TARGET_UTILIZATION*100:.0f}%)")

    # Build list of image sizes to try (requested first, then fallbacks)
    sizes_to_try = [imgsz] + [s for s in IMGSZ_FALLBACKS if s < imgsz]

    for try_sz in sizes_to_try:
        try:
            print(f"\n📊 Profiling {model_name} @ {try_sz}px...")
            base_mem, per_img_mem = _profile_model_memory(model_name, try_sz)
            print(f"   Base memory    : {base_mem:.2f} GB (model + optimizer, estimated)")
            print(f"   Per-image mem  : {per_img_mem:.3f} GB (activations + grads, estimated)")

            # Calculate optimal batch size
            available = target_vram - base_mem
            if available <= 0:
                print(f"   ⚠️  Model base memory ({base_mem:.2f} GB) exceeds target. Trying smaller imgsz.")
                continue

            optimal_batch = int(available / per_img_mem)
            batch = max(MIN_BATCH, min(MAX_BATCH, optimal_batch))

            # Calculate actual predicted usage
            predicted_usage = base_mem + batch * per_img_mem
            usage_pct = predicted_usage / total_vram * 100

            if batch >= MIN_BATCH:
                print(f"\n   ✅ Optimal batch size: {batch}")
                print(f"   Predicted VRAM usage: {predicted_usage:.2f} GB ({usage_pct:.1f}%)")
                if try_sz != imgsz:
                    print(f"   ⚠️  Reduced imgsz {imgsz} → {try_sz} to fit GPU.")
                return batch, try_sz
            else:
                print(f"   ⚠️  batch={optimal_batch} too small at {try_sz}px. Trying smaller imgsz.")

        except Exception as e:
            print(f"   ⚠️  Profiling failed at {try_sz}px: {e}")
            torch.cuda.empty_cache()
            gc.collect()
            continue

    # Absolute fallback — conservative estimate
    print("\n⚠️  All profiling attempts failed. Using conservative fallback.")
    print(f"   batch={MIN_BATCH}, imgsz=640")
    return MIN_BATCH, 640


# ── Quick test ────────────────────────────────────────────────────
bs_test, sz_test = estimate_batch_size("yolov8m.pt", 768)
print("\n" + "=" * 50)
print(f"  Final: batch={bs_test}, imgsz={sz_test}")
print("=" * 50)


🖥️  GPU: Tesla T4
   Total VRAM : 15.64 GB
   Target usage: 13.68 GB (88%)

📊 Profiling yolov8m.pt @ 768px...
   Base memory    : 0.50 GB (model + optimizer, estimated)
   Per-image mem  : 0.230 GB (activations + grads, estimated)

   ✅ Optimal batch size: 57
   Predicted VRAM usage: 13.62 GB (87.1%)

  Final: batch=57, imgsz=768


## 4 · Training Engine (OOM Recovery + Resume)

In [5]:
"""
Core training engine with:
  • Safe resume cascade: last.pt → best.pt → fresh (stage-aware)
  • FIXED checkpoint health validation (handles all YOLOv8 state_dict formats)
  • FIXED gradient clipping (on_before_optimizer_step — fires BEFORE optimizer)
  • NaN/Inf gradient sanitization via nan_to_num_ (handles AMP-produced NaN grads)
  • NaN loss early-stop guard (prevents writing corrupted last.pt)
  • Reduced lr0=5e-4 (was 1e-3) to prevent loss spikes
  • Extended warmup_epochs=10 (was 5) for large image sizes
  • CUDA OOM recovery (halve batch, then reduce imgsz)
  • Periodic checkpoint sync (survives session crashes)
  • Time-guard: auto-saves before session limit
  • Artifact copying (best.pt, last.pt, logs)
  • Structured logging: [RESUME MODE] [CHECKPOINT SAVED]
"""

PIPELINE_START_TIME = time.time()
CHECKPOINT_SYNC_INTERVAL = 5   # Sync checkpoints every N epochs


def elapsed_hours() -> float:
    """Hours since the pipeline started."""
    return (time.time() - PIPELINE_START_TIME) / 3600


def validate_checkpoint_health(ckpt_path) -> bool:
    if not ckpt_path.is_file():
        return False
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt.get("ema") or ckpt.get("model")
        if state is None:
            return True
        if hasattr(state, "state_dict"):
            state_dict = state.float().state_dict()
        elif isinstance(state, dict):
            state_dict = state   # ← THIS was the bug: old code crashed here
        else:
            return True
        bad_keys = []
        for name, tensor in state_dict.items():
            if not isinstance(tensor, torch.Tensor):
                continue
            if not tensor.is_floating_point():
                continue   # ← skip int tensors (num_batches_tracked etc.)
            if not torch.isfinite(tensor).all():
                bad_keys.append(name)
        if bad_keys:
            print(f"  ⚠️  Corrupted: {bad_keys[:3]}{'...' if len(bad_keys)>3 else ''}")
            return False
        return True
    except Exception as e:
        print(f"  ⚠️  Could not read {ckpt_path.name}: {e}")
        return True   # ← IO error ≠ NaN corruption, do NOT delete the file






def find_resume_checkpoint(run_name: str, model_dir: Path) -> tuple:
    """
    Stage-aware resume cascade with health validation:
      1. last.pt in run weights dir  -> resume=True (full state restore)
      2. last.pt in CHECKPOINTS_DIR (Colab synced copy) -> resume=True
      3. latest epoch*.pt in run weights dir -> resume=True
      4. best.pt in model_dir        -> load weights only (some epochs lost)
      5. Neither                     -> start fresh

    Returns: (checkpoint_path, use_resume_flag)
    """
    import re
    weights_dir = RUNS_DIR / run_name / "weights"

    # 1. Check run weights dir for last.pt
    last_pt = weights_dir / "last.pt"
    if last_pt.is_file():
        if validate_checkpoint_health(last_pt):
            print("=" * 60)
            print(f"  [RESUME MODE] Resuming from healthy last.pt (full state)")
            print(f"  Source: {last_pt}")
            print("=" * 60)
            return last_pt, True
        else:
            corrupted_path = last_pt.with_suffix(".pt.corrupted")
            print("=" * 60)
            print(f"  [RESUME MODE] last.pt is corrupted! Renaming to {corrupted_path.name}")
            print("=" * 60)
            try:
                shutil.move(str(last_pt), str(corrupted_path))
            except Exception:
                pass

    # 2. Check CHECKPOINTS_DIR for last.pt (synced copy)
    last_pt_model = CHECKPOINTS_DIR / "last.pt"
    if last_pt_model.is_file():
        if validate_checkpoint_health(last_pt_model):
            dst = weights_dir
            dst.mkdir(parents=True, exist_ok=True)
            try:
                shutil.copy2(last_pt_model, dst / "last.pt")
                args_src = model_dir / "args.yaml"
                if args_src.is_file():
                    shutil.copy2(args_src, RUNS_DIR / run_name / "args.yaml")
                print("=" * 60)
                print(f"  [RESUME MODE] Resuming from synced last.pt")
                print(f"  Source: {last_pt_model} → copied to {dst / 'last.pt'}")
                print("=" * 60)
                return dst / "last.pt", True
            except Exception:
                pass
        else:
            corrupted_path = last_pt_model.with_suffix(".pt.corrupted")
            print(f"  [RESUME MODE] {CHECKPOINTS_DIR.name}/last.pt is corrupted! Renaming.")
            try:
                shutil.move(str(last_pt_model), str(corrupted_path))
            except Exception:
                pass

    # 3. Fallback to scanning for epoch*.pt
    if weights_dir.is_dir():
        epoch_files = list(weights_dir.glob("epoch*.pt"))
        if epoch_files:
            epoch_data = []
            for f in epoch_files:
                match = re.search(r"epoch(\d+)\.pt$", f.name)
                if match:
                    epoch_data.append((int(match.group(1)), f))
            epoch_data.sort(key=lambda x: x[0], reverse=True)
            for epoch_num, epoch_pt in epoch_data:
                if validate_checkpoint_health(epoch_pt):
                    print("=" * 60)
                    print(f"  [RESUME MODE] Resuming from healthy epoch checkpoint: {epoch_pt.name}")
                    print(f"  Source: {epoch_pt}")
                    print("=" * 60)
                    return epoch_pt, True
                else:
                    corrupted_path = epoch_pt.with_suffix(".pt.corrupted")
                    print(f"  [RESUME MODE] {epoch_pt.name} is corrupted! Renaming.")
                    try:
                        shutil.move(str(epoch_pt), str(corrupted_path))
                    except Exception:
                        pass

    # 4. Check model_dir for best.pt (fallback — weight-only)
    best_pt = model_dir / "best.pt"
    if best_pt.is_file():
        if validate_checkpoint_health(best_pt):
            print("=" * 60)
            print(f"  [RESUME MODE] last.pt missing — using healthy best.pt weights")
            print(f"  Source: {best_pt}")
            print("  Note: Training restarts from best weights (some epochs lost)")
            print("=" * 60)
            return best_pt, False
        else:
            corrupted_path = best_pt.with_suffix(".pt.corrupted")
            print(f"  [RESUME MODE] {model_dir.name}/best.pt is corrupted! Renaming.")
            try:
                shutil.move(str(best_pt), str(corrupted_path))
            except Exception:
                pass

    # 5. No healthy checkpoint found
    print("=" * 60)
    print("  [RESUME MODE] No healthy checkpoint found — starting fresh")
    print("=" * 60)
    return None, False


def sync_checkpoints(run_name: str, model_dir: Path) -> None:
    """Copy best.pt and last.pt from run dir to persistent model dirs.
    Called periodically during training to survive session crashes."""
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights"
    if not weights.is_dir():
        return
    for src_name, dst_dir in [("best.pt", model_dir), ("last.pt", CHECKPOINTS_DIR)]:
        src = weights / src_name
        if src.is_file():
            try:
                shutil.copy2(src, dst_dir / src_name)
            except Exception as e:
                print(f"  ⚠️ Sync failed for {src_name}: {e}")
    results_csv = run_dir / "results.csv"
    if results_csv.is_file():
        try:
            shutil.copy2(results_csv, LOGS_DIR / f"{run_name}_results.csv")
            shutil.copy2(results_csv, model_dir / "results.csv")
        except Exception:
            pass
    args_yaml = run_dir / "args.yaml"
    if args_yaml.is_file():
        try:
            shutil.copy2(args_yaml, model_dir / "args.yaml")
        except Exception:
            pass
    print(f"  [CHECKPOINT SAVED] ✅ Synced to GDrive")


def copy_stage_artifacts(run_name: str, model_dir: Path) -> None:
    """Copy best/last weights and logs from a training run."""
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights"
    for src_name, dst_dir in [("best.pt", model_dir), ("last.pt", CHECKPOINTS_DIR)]:
        src = weights / src_name
        if src.is_file():
            shutil.copy2(src, dst_dir / src_name)
            print(f"  ✅ {src_name} → {dst_dir / src_name}")
    for pattern in ["*.csv", "*.png", "*.json"]:
        for f in run_dir.glob(pattern):
            shutil.copy2(f, LOGS_DIR / f"{run_name}_{f.name}")
    args_yaml = run_dir / "args.yaml"
    if args_yaml.is_file():
        shutil.copy2(args_yaml, model_dir / "args.yaml")
    print(f"  ✅ Logs → {LOGS_DIR}")


def recover_metrics_from_checkpoint(model_dir: Path, run_name: str, model_name: str) -> dict | None:
    """Recover metrics by validating a saved best.pt checkpoint."""
    best_pt = model_dir / "best.pt"
    if not best_pt.is_file():
        return None
    print(f"🔄 Recovering metrics from {best_pt}...")
    try:
        val_model = YOLO(str(best_pt))
        run_args_path = RUNS_DIR / run_name / "args.yaml"
        imgsz = 768
        if run_args_path.is_file():
            with open(run_args_path, "r") as f:
                run_args = yaml.safe_load(f)
            imgsz = run_args.get("imgsz", 768)
        metrics = val_model.val(
            data=str(DATA_YAML.resolve()), imgsz=imgsz, batch=8,
            device=0 if torch.cuda.is_available() else "cpu", verbose=False,
        )
        return {
            "stage": run_name, "model": model_name, "imgsz": imgsz,
            "mAP50": metrics.box.map50, "mAP50_95": metrics.box.map,
            "precision": metrics.box.mp, "recall": metrics.box.mr,
            "duration_h": -1,
        }
    except Exception as e:
        print(f"  ⚠️ Recovery failed: {e}")
        return None


def train_stage(
    model_name: str, imgsz: int, epochs: int, run_name: str,
    model_dir: Path, patience: int = 30, max_retries: int = 4,
) -> dict | None:
    """
    Train one stage with OOM recovery + time guard + NaN protection.

    On CUDA OOM: halve batch → reduce imgsz → skip.
    On NaN/Inf/corrupted: retry with clean fallback.
    Time guard: force-saves checkpoint if approaching session limit.
    """
    batch, actual_imgsz = estimate_batch_size(model_name, imgsz)
    stage_start = time.time()

    for attempt in range(1, max_retries + 1):
        print("\n" + "=" * 65)
        print(f"🚀 {run_name} | Attempt {attempt}/{max_retries} | "
              f"batch={batch} | imgsz={actual_imgsz}")
        print(f"   Pipeline elapsed: {elapsed_hours():.1f} h | "
              f"Session limit: {SESSION_TIME_LIMIT_H} h")
        print("=" * 65 + "\n")

        try:
            ckpt, use_resume = find_resume_checkpoint(run_name, model_dir)

            # ── FIX: Gradient clip callback on on_before_optimizer_step ──────
            # Previously registered on on_train_batch_end which fires AFTER
            # the optimizer step — making clipping useless. This fires BEFORE.
            def _grad_clip_callback(trainer):
                if hasattr(trainer, "model"):
                    for p in trainer.model.parameters():
                        if p.grad is not None:
                            torch.nan_to_num_(p.grad, nan=0.0, posinf=1e4, neginf=-1e4)
                    torch.nn.utils.clip_grad_norm_(
                        trainer.model.parameters(), max_norm=10.0
                    )

            # ── Epoch-end callback: NaN loss guard + sync + time guard ───────
            def _epoch_callback(trainer, _rn=run_name, _md=model_dir):
                # ── NaN loss guard ──────────────────────────────────
                loss = getattr(trainer, "loss", None)
                if loss is not None:
                    try:
                        if not torch.isfinite(torch.as_tensor(loss)).all():
                            print(f"\n🚨 [NaN GUARD] NaN/Inf loss at epoch "
                                  f"{trainer.epoch} — stopping before save!")
                            trainer.epoch = trainer.epochs  # forces clean exit
                            return  # do NOT sync — avoids writing corrupted last.pt
                    except Exception:
                        pass

                # FIX: NaN loss early-stop — prevents writing a corrupted last.pt
                # If loss is NaN/Inf, stop training immediately WITHOUT syncing
                loss = getattr(trainer, "loss", None)
                if loss is not None:
                    try:
                        loss_tensor = torch.as_tensor(loss)
                        if not torch.isfinite(loss_tensor).all():
                            print(f"\n🚨 NaN/Inf loss detected at epoch {trainer.epoch}!")
                            print("   Stopping immediately to prevent checkpoint corruption.")
                            trainer.epoch = trainer.epochs  # force YOLO to exit cleanly
                            return  # skip sync — do NOT write a bad checkpoint
                    except Exception:
                        pass

                # Periodic checkpoint sync (only when loss is healthy)
                if trainer.epoch % CHECKPOINT_SYNC_INTERVAL == 0 and trainer.epoch > 0:
                    sync_checkpoints(_rn, _md)

                # Time guard
                h = elapsed_hours()
                remaining = SESSION_TIME_LIMIT_H - h
                if remaining < 0.5:
                    print("\n" + "!" * 60)
                    print(f"  [TIME GUARD] ⚠️ Approaching session limit!")
                    print(f"  Elapsed: {h:.2f}h | Limit: {SESSION_TIME_LIMIT_H}h")
                    print(f"  Force-saving checkpoint and stopping...")
                    print("!" * 60)
                    sync_checkpoints(_rn, _md)
                    trainer.epoch = trainer.epochs  # Force stop
                elif remaining < 1.0:
                    print(f"\n  ⏰ Time warning: {remaining:.1f}h remaining before session limit")

            # ── FIXED training hyperparameters ───────────────────────────────
            # lr0=5e-4   (was 1e-3) — primary fix for NaN at epoch ~51
            #            AdamW + YOLOv8l/m at 768-832px overflows with lr=1e-3
            # lrf=0.005  (was 0.01) — proportionally reduced final LR
            # warmup_epochs=10 (was 5) — longer warmup stabilises large imgsz
            # warmup_bias_lr=0.01 — controls bias LR during warmup
            # clip_grad=10.0 — YOLOv8 built-in clip (belt-and-suspenders)
            TRAIN_KWARGS = dict(
                data=str(DATA_YAML.resolve()), imgsz=actual_imgsz,
                epochs=epochs, patience=patience, batch=batch,
                optimizer="AdamW",
                lr0=5e-4,            # FIX: was 1e-3 — too high, caused NaN ~epoch 51
                lrf=0.005,           # FIX: was 0.01 — proportionally reduced
                cos_lr=True,
                weight_decay=5e-4,
                warmup_epochs=10,    # FIX: was 5 — longer warmup for 768px+
                warmup_bias_lr=0.01,
                momentum=0.937,
                amp=True,
                clip_grad=10.0,      # FIX: YOLOv8 built-in gradient clip (backup)
                cache="ram", workers=4,
                device=0 if torch.cuda.is_available() else "cpu",
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                degrees=5.0, translate=0.1, scale=0.5,
                shear=2.0, flipud=0.2, fliplr=0.5,
                mosaic=1.0, mixup=0.1, copy_paste=0.1,
                project=str(RUNS_DIR), name=run_name, exist_ok=True,
                save_period=5, plots=True, verbose=True,
            )

            if ckpt and use_resume:
                # FULL RESUME: preserves epoch, optimizer, scheduler
                print("🔄 Full resume with resume=True")
                model = YOLO(str(ckpt))
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(resume=True)
            elif ckpt and not use_resume:
                # WEIGHT-ONLY RESUME: loads best.pt weights, fresh training args
                print("🔄 Loading best.pt weights (fresh training from epoch 0)")
                model = YOLO(str(ckpt))
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(**TRAIN_KWARGS)
            else:
                # FRESH START
                print("✨ Starting fresh training")
                model = YOLO(model_name)
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(**TRAIN_KWARGS)

            # ── Success ─────────────────────────────────────────────────────
            duration = (time.time() - stage_start) / 3600
            print(f"\n✅ {run_name} completed in {duration:.2f} hours.")
            copy_stage_artifacts(run_name, model_dir)

            log_entry = {
                "stage": run_name, "model": model_name, "imgsz": actual_imgsz,
                "batch": batch, "duration_hours": round(duration, 2),
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            log_file = LOGS_DIR / "training_log.json"
            logs = json.loads(log_file.read_text()) if log_file.is_file() else []
            logs.append(log_entry)
            log_file.write_text(json.dumps(logs, indent=2))

            best_pt = model_dir / "best.pt"
            if best_pt.is_file():
                val_model = YOLO(str(best_pt))
                metrics = val_model.val(
                    data=str(DATA_YAML.resolve()), imgsz=actual_imgsz,
                    batch=batch, device=0 if torch.cuda.is_available() else "cpu",
                    verbose=False,
                )
                return {
                    "stage": run_name, "model": model_name, "imgsz": actual_imgsz,
                    "mAP50": metrics.box.map50, "mAP50_95": metrics.box.map,
                    "precision": metrics.box.mp, "recall": metrics.box.mr,
                    "duration_h": round(duration, 2),
                }
            return None

        except RuntimeError as e:
            err_str = str(e).lower()
            if "out of memory" in err_str or ("cuda" in err_str and "nan" not in err_str):
                print(f"\n⚠️  CUDA OOM at batch={batch}, imgsz={actual_imgsz}")
                torch.cuda.empty_cache()
                gc.collect()
                if batch > MIN_BATCH:
                    batch = max(MIN_BATCH, batch // 2)
                    print(f"   → Reducing batch to {batch}")
                elif actual_imgsz > 640:
                    actual_imgsz -= 64
                    batch = MIN_BATCH
                    print(f"   → Reducing imgsz to {actual_imgsz}, batch={batch}")
                else:
                    print("   ❌ Cannot reduce further. Skipping stage.")
                    return None
            elif "corrupted" in err_str or "nan/inf" in err_str or "nan" in err_str:
                print(f"\n🚨 RuntimeError: {e}")
                print(f"   Checkpoint corruption or NaN loss detected.")
                print("   Retrying with clean fallback...")
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise

    print(f"\n❌ {run_name}: All {max_retries} attempts exhausted.")
    return None


print("✅ Training engine loaded (v2 — NaN-hardened).")
print(f"   Checkpoint sync every {CHECKPOINT_SYNC_INTERVAL} epochs")
print(f"   Time guard at {SESSION_TIME_LIMIT_H} hours")
print(f"   Gradient clipping: clip=10.0 (on_before_optimizer_step + nan_to_num_)")
print(f"   Learning rate: lr0=5e-4 (reduced from 1e-3 to prevent NaN)")
print(f"   Warmup: warmup_epochs=10 (extended from 5 for large imgsz)")
print(f"   Loss guard: NaN/Inf loss stops training before checkpoint write")
print(f"   Health check: handles all YOLOv8 state_dict formats (no false positives)")


✅ Training engine loaded.
   Checkpoint sync every 5 epochs
   Time guard at 11.5 hours


## 5 · Stage 1 — YOLOv8m (768px, 400 epochs)

In [6]:
import torch
print("Torch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO
YOLO("yolov8n.pt")
print("Everything working ✅")

Torch: 2.10.0+cu128
CUDA Available: True
GPU: Tesla T4
Everything working ✅


In [7]:
# Run this in a cell BEFORE starting training:
# import shutil
# old_run = RUNS_DIR / "pothole_v8m"
# if old_run.exists():
#     shutil.move(str(old_run), str(old_run) + "_old_batch2")
#     print(f"♻️ Moved old run to {old_run}_old_batch2")


In [8]:
"""
Stage 1: Train YOLOv8m at 768px.
On completion (full epochs or early stopping), proceeds to Stage 2.
"""

print("\n" + "━"*65)
print("  STAGE 1 — YOLOv8m @ 768px")
print("━"*65 + "\n")

metrics_v8m = train_stage(
    model_name = "yolov8m.pt",
    imgsz      = 768,
    epochs     = 400,
    run_name   = "pothole_v8m",
    model_dir  = V8M_DIR,
    patience   = 30,
)

if metrics_v8m:
    print("\n📊 Stage 1 Results:")
    for k, v in metrics_v8m.items():
        print(f"  {k:12s}: {v}")
else:
    print("\n⚠️  Stage 1 did not return metrics.")

# Free GPU memory before Stage 2
torch.cuda.empty_cache()
gc.collect()
print(f"\n⏱️  Total pipeline time: {elapsed_hours():.1f} h")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STAGE 1 — YOLOv8m @ 768px
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🖥️  GPU: Tesla T4
   Total VRAM : 15.64 GB
   Target usage: 13.68 GB (88%)

📊 Profiling yolov8m.pt @ 768px...
   Base memory    : 0.50 GB (model + optimizer, estimated)
   Per-image mem  : 0.231 GB (activations + grads, estimated)

   ✅ Optimal batch size: 57
   Predicted VRAM usage: 13.68 GB (87.5%)

🚀 pothole_v8m | Attempt 1/4 | batch=57 | imgsz=768
   Pipeline elapsed: 0.0 h | Session limit: 11.5 h

  [RESUME MODE] Resuming from last.pt
  Source: /content/drive/MyDrive/pothole_detection_system/backend/runs/pothole_v8m/weights/last.pt
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=26, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None,

RuntimeError: Checkpoint /content/drive/MyDrive/pothole_detection_system/backend/runs/pothole_v8m/weights/last.pt is corrupted with NaN/Inf weights

## 6 · Stage 2 — YOLOv8l (832px, 400 epochs)

In [ ]:
"""
Stage 2: Train YOLOv8l at 832px (or highest that fits GPU).
Automatically starts after Stage 1 finishes.
"""

print("\n" + "━"*65)
print("  STAGE 2 — YOLOv8l @ 832px")
print("━"*65 + "\n")

metrics_v8l = train_stage(
    model_name = "yolov8l.pt",
    imgsz      = 832,
    epochs     = 400,
    run_name   = "pothole_v8l",
    model_dir  = V8L_DIR,
    patience   = 30,
)

if metrics_v8l:
    print("\n📊 Stage 2 Results:")
    for k, v in metrics_v8l.items():
        print(f"  {k:12s}: {v}")
else:
    print("\n⚠️  Stage 2 did not return metrics.")

torch.cuda.empty_cache()
gc.collect()
print(f"\n⏱️  Total pipeline time: {elapsed_hours():.1f} h")

## 7 · Time-Aware Bonus Fine-Tuning

In [ ]:
"""
If total runtime < SESSION_TIME_LIMIT_H and both models finished,
continue fine-tuning YOLOv8l at higher resolution.
"""

metrics_v8l = globals().get("metrics_v8l", None)

hours_elapsed = elapsed_hours()
print(f"⏱️  Time elapsed: {hours_elapsed:.1f} h  |  Session limit: {SESSION_TIME_LIMIT_H} h")

remaining = SESSION_TIME_LIMIT_H - hours_elapsed
if remaining > 1.5:
    bonus_epochs = max(50, int(remaining * 8))
    bonus_epochs = min(bonus_epochs, 200)

    print(f"\n🔥 Bonus stage: fine-tuning YOLOv8l for ~{bonus_epochs} more epochs.")
    print(f"   Estimated remaining time: {remaining:.1f} h\n")

    v8l_best = V8L_DIR / "best.pt"
    if v8l_best.is_file():
        metrics_bonus = train_stage(
            model_name=str(v8l_best), imgsz=896, epochs=bonus_epochs,
            run_name="pothole_v8l_finetune", model_dir=V8L_DIR, patience=20,
        )
        if metrics_bonus:
            print("\n📊 Bonus Fine-Tune Results:")
            for k, v in metrics_bonus.items():
                print(f"  {k:12s}: {v}")
            if metrics_v8l and metrics_bonus.get("mAP50_95", 0) > metrics_v8l.get("mAP50_95", 0):
                metrics_v8l = metrics_bonus
                print("\n✅ Bonus stage improved v8l — updated metrics.")
    else:
        print("⚠️  No v8l best.pt found for bonus fine-tuning.")
else:
    print("ℹ️  Skipping bonus stage — insufficient time remaining.")

print(f"\n⏱️  Total pipeline time: {elapsed_hours():.1f} h")

## 8 · Model Comparison & Best Selection

In [ ]:
"""
Compare v8m vs v8l and automatically copy the best one to models/final/best.pt.
If metrics are not available (e.g., session crashed), auto-recover from saved checkpoints.
"""

print("\n" + "="*65)
print("  MODEL COMPARISON")
print("="*65 + "\n")

# ── Safely access metrics (may be undefined after a session crash) ──
metrics_v8m = globals().get("metrics_v8m", None)
metrics_v8l = globals().get("metrics_v8l", None)

# ── Auto-recover: If checkpoints exist on GDrive but metrics are missing,
#    run validation to recover them ──
if metrics_v8m is None and (V8M_DIR / "best.pt").is_file():
    print("⚠️  metrics_v8m not available — recovering from saved checkpoint...")
    metrics_v8m = recover_metrics_from_checkpoint(V8M_DIR, "pothole_v8m", "yolov8m.pt")

if metrics_v8l is None and (V8L_DIR / "best.pt").is_file():
    print("⚠️  metrics_v8l not available — recovering from saved checkpoint...")
    metrics_v8l = recover_metrics_from_checkpoint(V8L_DIR, "pothole_v8l", "yolov8l.pt")

all_metrics = []
if metrics_v8m:
    all_metrics.append(metrics_v8m)
if metrics_v8l:
    all_metrics.append(metrics_v8l)

if not all_metrics:
    print("❌ No metrics available from any stage.")
    print("   Ensure at least one stage has completed and best.pt exists in:")
    print(f"     V8M: {V8M_DIR / 'best.pt'} (exists: {(V8M_DIR / 'best.pt').is_file()})")
    print(f"     V8L: {V8L_DIR / 'best.pt'} (exists: {(V8L_DIR / 'best.pt').is_file()})")
else:
    # Display comparison table
    header = f"{'Metric':>15s} |"
    for m in all_metrics:
        header += f" {m['stage']:>18s} |"
    print(header)
    print("-" * len(header))

    for key in ["mAP50", "mAP50_95", "precision", "recall", "duration_h", "imgsz"]:
        row = f"{key:>15s} |"
        for m in all_metrics:
            val = m.get(key, "N/A")
            if isinstance(val, float):
                row += f" {val:>18.4f} |"
            else:
                row += f" {str(val):>18s} |"
        print(row)

    # Select best by mAP50-95
    best = max(all_metrics, key=lambda x: x.get("mAP50_95", 0))
    print(f"\n🏆 Best model: {best['stage']} (mAP50-95: {best['mAP50_95']:.4f})")

    # Copy best model to final/
    if "v8m" in best["stage"]:
        src = V8M_DIR / "best.pt"
    else:
        src = V8L_DIR / "best.pt"

    if src.is_file():
        shutil.copy2(src, FINAL_DIR / "best.pt")
        print(f"✅ Best model copied → {FINAL_DIR / 'best.pt'}")

    # Save comparison JSON
    comp_file = LOGS_DIR / "model_comparison.json"
    comp_file.write_text(json.dumps(all_metrics, indent=2))
    print(f"📄 Comparison saved → {comp_file}")

# ── Set BEST_MODEL_PATH for downstream cells ──
BEST_MODEL_PATH = FINAL_DIR / "best.pt"
if not BEST_MODEL_PATH.is_file():
    for d in [V8L_DIR, V8M_DIR]:
        if (d / "best.pt").is_file():
            BEST_MODEL_PATH = d / "best.pt"
            break
print(f"\n🏆 BEST_MODEL_PATH = {BEST_MODEL_PATH} (exists: {BEST_MODEL_PATH.is_file()})")


## 9 · Evaluation Visualizations

In [ ]:
"""
Plot training curves for both stages.
"""

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Training Curves", fontsize=16, fontweight="bold")

plot_cols = [
    ("train/box_loss", "Box Loss"), ("train/cls_loss", "Cls Loss"),
    ("train/dfl_loss", "DFL Loss"), ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)", "Recall"), ("metrics/mAP50(B)", "mAP@50"),
]

colors = {"pothole_v8m": "#2196F3", "pothole_v8l": "#FF5722"}

for run_name, color in colors.items():
    csv_path = RUNS_DIR / run_name / "results.csv"
    if not csv_path.is_file():
        continue
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    for ax, (col, title) in zip(axes.flat, plot_cols):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], label=run_name, color=color, linewidth=1.2)
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(LOGS_DIR / "combined_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📈 Saved → {LOGS_DIR / 'combined_training_curves.png'}")

In [ ]:
"""
Display confusion matrix from the best model validation.
"""

for run_name in ["pothole_v8l", "pothole_v8m"]:
    cm_path = RUNS_DIR / run_name / "confusion_matrix_normalized.png"
    if cm_path.is_file():
        print(f"📊 Confusion Matrix — {run_name}")
        display(IPImage(filename=str(cm_path), width=700))
        break

# Also check val run directories
for vd in sorted(RUNS_DIR.glob("val*")):
    cm = vd / "confusion_matrix_normalized.png"
    if cm.is_file():
        print(f"📊 Confusion Matrix — {vd.name}")
        display(IPImage(filename=str(cm), width=700))
        break

In [ ]:
"""
Sample predictions from the best model.
"""

BEST_MODEL_PATH = FINAL_DIR / "best.pt"
if not BEST_MODEL_PATH.is_file():
    for d in [V8L_DIR, V8M_DIR]:
        if (d / "best.pt").is_file():
            BEST_MODEL_PATH = d / "best.pt"
            break

if BEST_MODEL_PATH.is_file():
    best_model = YOLO(str(BEST_MODEL_PATH))
    val_imgs = sorted((DATASET_DIR / "images" / "val").glob("*.*"))[:6]

    if val_imgs:
        results = best_model.predict(
            source=[str(p) for p in val_imgs],
            imgsz=768, conf=0.25,
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False,
        )

        fig, axes = plt.subplots(2, 3, figsize=(20, 13))
        fig.suptitle("Sample Predictions (Best Model)", fontsize=16, fontweight="bold")

        for ax, result in zip(axes.flat, results):
            ann = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
            ax.imshow(ann)
            ax.axis("off")
            n = len(result.boxes)
            ax.set_title(f"{Path(result.path).name} ({n} det)", fontsize=10)

        plt.tight_layout()
        fig.savefig(LOGS_DIR / "sample_predictions.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("⚠️  No best.pt found for predictions.")

## 10 · Inference

### 10a · Image Inference

In [ ]:
"""
Run inference on a single image.
Set IMAGE_PATH below, then run this cell.
"""

IMAGE_PATH = "../dataset_768/images/test/example.jpg"   # ← change this


def predict_image(image_path: str, conf: float = 0.25) -> None:
    img_path = Path(image_path)
    if not img_path.is_file():
        print(f"❌ Not found: {img_path}")
        return

    model = YOLO(str(BEST_MODEL_PATH))
    results = model.predict(
        source=str(img_path), imgsz=768, conf=conf,
        device=0 if torch.cuda.is_available() else "cpu", verbose=False,
    )
    result = results[0]
    annotated = result.plot()

    out_path = INFERENCE_DIR / f"pred_{img_path.stem}.jpg"
    cv2.imwrite(str(out_path), annotated)

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{img_path.name} — {len(result.boxes)} detection(s)", fontsize=14)
    plt.show()

    for i, box in enumerate(result.boxes):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print(f"  [{i+1}] pothole  conf={box.conf.item():.3f}  "
              f"bbox=[{x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f}]")
    print(f"💾 Saved → {out_path}")


predict_image(IMAGE_PATH)

### 10b · Video Inference

In [ ]:
"""
Run inference on a video file.
Set VIDEO_PATH below, then run this cell.
"""

VIDEO_PATH = "../test_video.mp4"   # ← change this


def predict_video(video_path: str, conf: float = 0.25) -> None:
    vid_path = Path(video_path)
    if not vid_path.is_file():
        print(f"❌ Not found: {vid_path}")
        return

    model = YOLO(str(BEST_MODEL_PATH))
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened():
        print(f"❌ Cannot open: {vid_path}")
        return

    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = INFERENCE_DIR / f"pred_{vid_path.stem}.mp4"
    writer   = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    print(f"🎬 Processing: {vid_path.name} ({w}×{h}, {fps:.0f} fps, {total} frames)")

    count = 0
    t0 = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        count += 1
        results = model.predict(source=frame, imgsz=768, conf=conf,
                                device=0 if torch.cuda.is_available() else "cpu",
                                verbose=False)
        ann = results[0].plot()
        cur_fps = count / (time.time() - t0) if time.time() > t0 else 0
        cv2.putText(ann, f"FPS: {cur_fps:.1f}  Det: {len(results[0].boxes)}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        writer.write(ann)
        if count % 100 == 0:
            print(f"   Frame {count}/{total} ({count/total*100:.0f}%)  FPS: {cur_fps:.1f}")

    cap.release()
    writer.release()
    avg_fps = count / (time.time() - t0)
    print(f"\n✅ Done! {count} frames | Avg FPS: {avg_fps:.1f} | 💾 {out_path}")


predict_video(VIDEO_PATH)

### 10c · Live Webcam Inference

In [ ]:
"""
Real-time webcam detection. Press q to quit, s to save a frame.
NOTE: Requires a display — won't work on headless servers.
"""


def live_webcam(camera_id: int = 0, conf: float = 0.30) -> None:
    model = YOLO(str(BEST_MODEL_PATH))
    cap   = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("❌ Cannot open webcam.")
        return

    print("🎥 Webcam started. Press q=quit, s=save.")
    n = 0
    t0 = time.time()
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            n += 1
            results = model.predict(source=frame, imgsz=768, conf=conf,
                                    device=0 if torch.cuda.is_available() else "cpu",
                                    verbose=False)
            ann = results[0].plot()
            fps = n / (time.time() - t0) if time.time() > t0 else 0
            cv2.putText(ann, f"FPS: {fps:.1f} | Potholes: {len(results[0].boxes)}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            cv2.imshow("Pothole Detection — Live", ann)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("s"):
                sp = INFERENCE_DIR / f"webcam_{n}.jpg"
                cv2.imwrite(str(sp), ann)
                print(f"📸 {sp}")
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print(f"🎥 Stopped. {n} frames.")


# Uncomment to start:
# live_webcam(camera_id=0)

## 11 · GPS Map Tagging

In [ ]:
"""
GPS-tagged pothole logging for map integration.

Outputs:
  • pothole_detections.json  — array of detection records
  • pothole_detections.geojson — for Leaflet / Google Maps / OSM
"""

POTHOLE_LOG = INFERENCE_DIR / "pothole_detections.json"


def load_log() -> list:
    return json.loads(POTHOLE_LOG.read_text()) if POTHOLE_LOG.is_file() else []


def save_log(log: list) -> None:
    POTHOLE_LOG.write_text(json.dumps(log, indent=2))


def log_pothole(lat: float, lon: float, conf: float, img_path: str) -> dict:
    entry = {
        "timestamp":  datetime.now(timezone.utc).isoformat(),
        "latitude":   round(lat, 6),
        "longitude":  round(lon, 6),
        "confidence": round(conf, 4),
        "image_path": str(img_path),
    }
    log = load_log()
    log.append(entry)
    save_log(log)
    print(f"📌 Logged @ ({lat}, {lon}) conf={conf:.3f}")
    return entry


def detect_and_log(image_path: str, lat: float, lon: float, conf: float = 0.25) -> list:
    img_p = Path(image_path)
    if not img_p.is_file():
        print(f"❌ {img_p}")
        return []
    model = YOLO(str(BEST_MODEL_PATH))
    results = model.predict(source=str(img_p), imgsz=768, conf=conf,
                            device=0 if torch.cuda.is_available() else "cpu", verbose=False)
    r = results[0]
    if len(r.boxes) == 0:
        print("ℹ️  No potholes detected.")
        return []
    out = INFERENCE_DIR / f"gps_{img_p.stem}.jpg"
    cv2.imwrite(str(out), r.plot())
    entries = []
    for box in r.boxes:
        entries.append(log_pothole(lat, lon, box.conf.item(), str(out.relative_to(BACKEND_DIR))))
    return entries


def export_geojson(out: Path | None = None) -> Path:
    log = load_log()
    features = [{"type": "Feature",
                 "geometry": {"type": "Point", "coordinates": [e["longitude"], e["latitude"]]},
                 "properties": {k: e[k] for k in ["timestamp", "confidence", "image_path"]}}
                for e in log]
    geo = {"type": "FeatureCollection", "features": features}
    p = out or INFERENCE_DIR / "pothole_detections.geojson"
    p.write_text(json.dumps(geo, indent=2))
    print(f"🗺️  GeoJSON: {p} ({len(features)} features)")
    return p


print("✅ Map tagging module loaded.")
print(f"   Log: {POTHOLE_LOG}")

In [ ]:
"""
Example usage (uncomment to use):
"""

# entries = detect_and_log(
#     image_path = "../dataset_768/images/test/some_image.jpg",
#     lat = 18.5204,
#     lon = 73.8567,
# )
#
# export_geojson()
# print(json.dumps(load_log(), indent=2))

## 12 · Utilities

In [ ]:
"""
Export best model to ONNX for deployment.
"""

def export_onnx(model_path: Path = BEST_MODEL_PATH) -> None:
    m = YOLO(str(model_path))
    m.export(format="onnx", imgsz=768, half=True, simplify=True)
    print(f"✅ ONNX exported next to {model_path}")

# Uncomment: export_onnx()

In [ ]:
"""
Final pipeline summary.
"""

# ── Safe access to BEST_MODEL_PATH and elapsed_hours ──
if "BEST_MODEL_PATH" not in dir():
    BEST_MODEL_PATH = FINAL_DIR / "best.pt"
    if not BEST_MODEL_PATH.is_file():
        for d in [V8L_DIR, V8M_DIR]:
            if (d / "best.pt").is_file():
                BEST_MODEL_PATH = d / "best.pt"
                break

try:
    total_h = elapsed_hours()
except NameError:
    total_h = -1  # unknown (session restarted)

print("\n" + "="*65)
print("  🏁 PIPELINE COMPLETE")
print("="*65)
print(f"  Total time      : {total_h:.2f} hours" if total_h >= 0 else "  Total time      : unknown (session restarted)")
print(f"  Best model      : {BEST_MODEL_PATH}")
print(f"  Final dir       : {FINAL_DIR}")
print(f"  Inference dir   : {INFERENCE_DIR}")
print(f"  Logs dir        : {LOGS_DIR}")

# Dataset counts
for split in ["train", "val", "test"]:
    d = DATASET_DIR / "images" / split
    n = len(list(d.glob("*"))) if d.is_dir() else 0
    print(f"  Dataset {split:>5}   : {n:,} images")

# Training log
log_file = LOGS_DIR / "training_log.json"
if log_file.is_file():
    print("\n  Training stages:")
    for entry in json.loads(log_file.read_text()):
        print(
            f"    {entry['stage']:25s} | "
            f"{entry['duration_hours']:.2f} h | "
            f"imgsz={entry['imgsz']} batch={entry['batch']}"
        )

print("="*65)
